# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AyeshaShahid-creator/Internship-flyrank-ML/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [2]:
import pandas as pd

df = pd.read_csv("https://raw.githubusercontent.com/AyeshaShahid-creator/Internship-flyrank-ML/main/data/raw/content_refresh_anonymized.csv")
staleness_check = df.groupby("freshness_tier").agg(
    n=("content_id", "count"),
    decline_rate=("trend_direction", lambda x: (x == "down").mean() * 100)
).round(1)

print(staleness_check)
position_check = df.groupby("position_tier").agg(
    n=("content_id", "count"),
    avg_ctr=("ctr", "mean")
).round(2)

order = ["top_3", "page_1", "striking", "page_3_5", "deep", "no_data"]
position_check = position_check.reindex([o for o in order if o in position_check.index])

print(position_check)


                    n  decline_rate
freshness_tier                     
0-30            20480          51.1
181+              174          47.1
31-90             175          58.9
91-180           9171          61.1
                   n  avg_ctr
position_tier                
top_3           2321     1.48
page_1         11814     0.65
striking        7304     0.32
page_3_5        7242     0.22
deep            1319     0.15


Staleness check: decline rate rises from 51.1% (0-30 days) to 61.1% (91-180 days), consistent with the refresh-flag logic. The 31-90 and 181+ buckets have very small n (175, 174) and shouldn't be trusted — most data sits in 0-30 and 91-180. Verdict: MIXED — directionally supportive but not clean across all tiers.
Position check: avg CTR falls monotonically from 1.48% (top_3) to 0.15% (deep) — every tier confirms the pattern, and n is solid across all tiers (min 1,319). Verdict: CONFIRMED.


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

Rule: score = 0.7 × (tier-average CTR minus page's CTR) + 0.3 × normalized staleness. Volume floor: impressions_90d ≥ 100. Reason code: CTR_GAP_VS_POSITION_TIER. Top 10% by score get REVIEW_FOR_REFRESH, rest get MONITOR.

In [4]:
import os

# 1. Only score pages with enough real traffic to matter (volume floor)
measurable = df[df["impressions_90d"] >= 100].copy()

# 2. Compute each position tier's average CTR (this is what you already found in Step 1b)
tier_avg_ctr = measurable.groupby("position_tier")["ctr"].transform("mean")

# 3. The core signal: how far below its tier's average CTR is this page?
measurable["ctr_gap"] = tier_avg_ctr - measurable["ctr"]

# Normalize ctr_gap to 0-1 (min-max scaling)
ctr_gap_norm = (measurable["ctr_gap"] - measurable["ctr_gap"].min()) / \
               (measurable["ctr_gap"].max() - measurable["ctr_gap"].min())

# Normalize staleness to 0-1
staleness_norm = (measurable["days_since_last_update"] - measurable["days_since_last_update"].min()) / \
                 (measurable["days_since_last_update"].max() - measurable["days_since_last_update"].min())

# 5. The score — CTR gap weighted heavier (0.7) since it's CONFIRMED, staleness lighter (0.3) since it's MIXED
measurable["action_score"] = (0.7 * ctr_gap_norm) + (0.3 * staleness_norm)

# 6. One reason code — this rule's whole reason for flagging
measurable["reason_code"] = "CTR_GAP_VS_POSITION_TIER"

# 7. One action label based on score rank
measurable["action_label"] = "MONITOR"
measurable.loc[measurable["action_score"] > measurable["action_score"].quantile(0.90), "action_label"] = "REVIEW_FOR_REFRESH"

# 8. Rank and select output columns
queue = measurable.sort_values("action_score", ascending=False)
output_cols = ["content_id", "client_id", "action_score", "reason_code", "action_label",
               "ctr", "avg_position", "position_tier", "days_since_last_update", "impressions_90d"]
queue = queue[output_cols]

# 9. Write the CSV — create the folder if it doesn't exist yet
os.makedirs("work/outputs", exist_ok=True)
queue.to_csv("work/outputs/baseline_action_score.csv", index=False)

print(f"Wrote {len(queue)} rows.")
print(queue.head(10))


Wrote 22006 rows.
                 content_id          client_id  action_score  \
21984  content_02b0d6e30129  client_19581e27de      1.000000   
3723   content_f488400fca67  client_9400f1b21c      0.992233   
1147   content_ab27c30d81f4  client_4ec9599fc2      0.991262   
15790  content_6476d1d8c050  client_19581e27de      0.982182   
9346   content_d25a099b3726  client_19581e27de      0.974415   
7452   content_72496874f806  client_4ec9599fc2      0.974064   
26840  content_7f116ae1f6f5  client_9400f1b21c      0.963350   
23741  content_df1fa766cac2  client_9400f1b21c      0.962095   
1227   content_4f241bad48a3  client_6208ef0f77      0.919351   
3651   content_fd16e3475c29  client_d029fa3a95      0.873786   

                    reason_code        action_label   ctr  avg_position  \
21984  CTR_GAP_VS_POSITION_TIER  REVIEW_FOR_REFRESH  0.00           6.9   
3723   CTR_GAP_VS_POSITION_TIER  REVIEW_FOR_REFRESH  0.00           5.7   
1147   CTR_GAP_VS_POSITION_TIER  REVIEW_FOR_REFRESH 

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

Top 10 review

content_02b0d6e30129 — page_1, 176 impressions, 0% CTR, not updated in 313 days. Flagged because it's ranking fine but getting no clicks at all. Could be wrong if this is a branded search where people don't need to click through anyway.

content_f488400fca67 — page_1, 155 impressions, 0% CTR, 305 days stale. Same pattern as above. Could be wrong if the keyword is still new and just hasn't picked up trust yet.

content_ab27c30d81f4 — page_1, only 103 impressions, 0% CTR. Flagged, but 103 impressions over 90 days is a small sample — a couple of real clicks next month could change this completely.

content_6476d1d8c050 — deep position (67.8), 304 impressions, 0% CTR. Could be wrong because deep-ranked pages naturally get almost no clicks anyway (~0.15% average) — this might just be restating that it's ranked badly, not a real content problem.

content_d25a099b3726 — deep position (64.5), 202 impressions, 0% CTR, 305 days stale. Same caveat as #4.

content_72496874f806 — page_1, 821 impressions, 0.24% CTR. This one feels like the strongest real case — good traffic volume and a real gap below the page_1 average (0.65%). Could be wrong if the content doesn't match what the searcher actually wants (e.g. wrong intent), since refreshing it wouldn't fix that.

content_7f116ae1f6f5 — page_1, 954 impressions, 0.42% CTR. Getting closer to average CTR, so this might mostly be here because of the staleness tie-break, not a big real gap.

content_df1fa766cac2 — page_1, 206 impressions, 0.49% CTR. Similar to #7 — CTR isn't that far below average.

content_4f241bad48a3 — striking position, 285 impressions, 0% CTR. Could be wrong because "striking distance" pages are borderline-ranked (position ~11-20) — this might need a ranking push more than a content refresh.

content_fd16e3475c29 — page_1, 429 impressions, 0% CTR, only 183 days stale (least stale in the whole top 10). This one got flagged almost purely on the CTR gap, which is good — shows the main signal working on its own, not just staleness dragging it up


In [6]:
top10 = queue.head(10)
print(top10[["content_id", "action_score", "position_tier", "ctr", "avg_position", "days_since_last_update", "impressions_90d"]].to_string(index=False))

          content_id  action_score position_tier  ctr  avg_position  days_since_last_update  impressions_90d
content_02b0d6e30129      1.000000        page_1 0.00           6.9                     313              176
content_f488400fca67      0.992233        page_1 0.00           5.7                     305              155
content_ab27c30d81f4      0.991262        page_1 0.00           8.9                     304              103
content_6476d1d8c050      0.982182          deep 0.00          67.8                     313              304
content_d25a099b3726      0.974415          deep 0.00          64.5                     305              202
content_72496874f806      0.974064        page_1 0.24           5.8                     301              821
content_7f116ae1f6f5      0.963350        page_1 0.42           9.0                     301              954
content_df1fa766cac2      0.962095        page_1 0.49           4.8                     304              206
content_4f241bad48a

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks: Rows #4 and #5 (deep-position pages) are the weakest in the top 10 — deep pages average ~0.15% CTR anyway, so flagging them as "underperforming" may just restate that they're ranked badly, not point to a fixable content problem. Rows #7 and #8 are also borderline — their CTR is close to their tier average, so they're likely in the top 10 mostly due to the staleness tie-break rather than a strong real gap.

Leakage check: Confirmed no use of trend_direction or trend_pct anywhere in the score, reason code, or action label — these are the label source and were excluded from all feature columns.

In [7]:
# Leakage check — confirm forbidden columns were never used as features
forbidden_cols = ["trend_direction", "trend_pct"]
used_cols = ["ctr", "avg_position", "position_tier", "days_since_last_update", "impressions_90d"]

leak_found = [c for c in forbidden_cols if c in used_cols]
print("Leakage check — forbidden columns used as features:", leak_found if leak_found else "None. Clean.")

# Also confirm the queue output doesn't carry the label columns
print("\nColumns in final output:", list(queue.columns))


Leakage check — forbidden columns used as features: None. Clean.

Columns in final output: ['content_id', 'client_id', 'action_score', 'reason_code', 'action_label', 'ctr', 'avg_position', 'position_tier', 'days_since_last_update', 'impressions_90d']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.